# Silver → Gold Data Quality (cross-warehouse)

**Source tables:** `HP_03_WH_HealthAnalytics` (`dim.*`, `fact.*`) — **read-only**.  
**DQ run/results storage:** `HP_09_WH_DataQuality` (`dq.SilverToGoldDQRun`, `dq.SilverToGoldDQResult`) — **writes here only**.

**Fabric setup:** This is a **T-SQL notebook**. Add both warehouses as data sources and set **HP_09_WH_DataQuality as the primary warehouse** before running any cell. The CREATE SCHEMA and CREATE TABLE statements use the primary warehouse; the checks use three-part source table names. Both warehouses must be accessible in the same workspace for these names to resolve; otherwise check Fabric cross-warehouse connectivity/permissions.

The repeated `FactCondition` DQ section was removed to avoid inserting duplicate checks.


In [1]:
IF NOT EXISTS (
    SELECT 1
    FROM sys.schemas
    WHERE name = 'dq'
)
BEGIN
    EXEC('CREATE SCHEMA dq');
END

In [2]:
IF OBJECT_ID('dq.SilverToGoldDQRun', 'U') IS NULL
BEGIN
CREATE TABLE dq.SilverToGoldDQRun
(
    DQRunID      VARCHAR(100) NOT NULL,
    StartTime    DATETIME2(6) NOT NULL,
    EndTime      DATETIME2(6) NULL,
    Status       VARCHAR(20)  NOT NULL
);
END;

In [3]:
IF OBJECT_ID('dq.SilverToGoldDQResult', 'U') IS NULL
BEGIN
    CREATE TABLE dq.SilverToGoldDQResult
    (
        DQResultKey BIGINT NOT NULL,

        DQRunID VARCHAR(100) NOT NULL,
        DQRunTime DATETIME2(6) NOT NULL,

        LayerName VARCHAR(50) NOT NULL,
        EntityName VARCHAR(100) NOT NULL,

        CheckName VARCHAR(500) NOT NULL,
        CheckType VARCHAR(100) NOT NULL,

        Severity VARCHAR(20) NOT NULL,
        Status VARCHAR(20) NOT NULL,

        FailedCount BIGINT NULL,
        TotalCount BIGINT NULL,

        Details VARCHAR(2000) NULL
    );
END;

#### Start the DQ run

In [4]:
-- ============================================================
-- START SILVER -> GOLD DQ RUN
-- ============================================================

-- Close any old RUNNING run
UPDATE [HP_09_WH_DataQuality].[dq].SilverToGoldDQRun
SET
    EndTime = SYSUTCDATETIME(),
    Status = 'ABANDONED'
WHERE Status = 'RUNNING';


-- Start new run
INSERT INTO [HP_09_WH_DataQuality].[dq].SilverToGoldDQRun
(
    DQRunID,
    StartTime,
    Status
)
SELECT
    CONVERT(VARCHAR(100), NEWID()),
    SYSUTCDATETIME(),
    'RUNNING';


SELECT TOP 1 *
FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQRun
ORDER BY StartTime DESC;

##### DimDate

In [5]:
-- ============================================================
-- DQ: DimDate
-- ============================================================

WITH CurrentRun AS
(
    SELECT TOP 1 DQRunID
    FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQRun
    WHERE Status = 'RUNNING'
    ORDER BY StartTime DESC
),

BaseKey AS
(
    SELECT
        ISNULL(MAX(DQResultKey), 0) AS BaseKey
    FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQResult
),

Checks AS
(
    -- DateKey null
    SELECT
        'DateKey populated' AS CheckName,
        'COMPLETENESS' AS CheckType,
        'CRITICAL' AS Severity,

        SUM(
            CASE WHEN DateKey IS NULL
                 THEN CAST(1 AS BIGINT)
                 ELSE CAST(0 AS BIGINT)
            END
        ) AS FailedCount,

        COUNT_BIG(*) AS TotalCount,

        'DateKey must not be NULL.' AS Details

    FROM [HP_03_WH_HealthAnalytics].dim.DimDate


    UNION ALL


    -- Duplicate DateKey
    SELECT
        'DateKey unique',
        'UNIQUENESS',
        'CRITICAL',

        (
            SELECT COUNT_BIG(*)
            FROM
            (
                SELECT DateKey
                FROM [HP_03_WH_HealthAnalytics].dim.DimDate
                GROUP BY DateKey
                HAVING COUNT_BIG(*) > 1
            ) d
        ),

        (SELECT COUNT_BIG(*) FROM [HP_03_WH_HealthAnalytics].dim.DimDate),

        'Each DateKey must occur once.'


    UNION ALL


    -- Duplicate FullDate
    SELECT
        'FullDate unique',
        'UNIQUENESS',
        'HIGH',

        (
            SELECT COUNT_BIG(*)
            FROM
            (
                SELECT FullDate
                FROM [HP_03_WH_HealthAnalytics].dim.DimDate
                GROUP BY FullDate
                HAVING COUNT_BIG(*) > 1
            ) d
        ),

        (SELECT COUNT_BIG(*) FROM [HP_03_WH_HealthAnalytics].dim.DimDate),

        'Each calendar date should occur once.'
),

Numbered AS
(
    SELECT
        *,
        ROW_NUMBER() OVER
        (
            ORDER BY CheckName
        ) AS rn

    FROM Checks
)

INSERT INTO [HP_09_WH_DataQuality].[dq].SilverToGoldDQResult
(
    DQResultKey,
    DQRunID,
    DQRunTime,
    LayerName,
    EntityName,
    CheckName,
    CheckType,
    Severity,
    Status,
    FailedCount,
    TotalCount,
    Details
)

SELECT
    b.BaseKey + n.rn,
    r.DQRunID,
    SYSUTCDATETIME(),

    'Gold',
    'DimDate',

    n.CheckName,
    n.CheckType,
    n.Severity,

    CASE
        WHEN ISNULL(n.FailedCount,0) = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END,

    ISNULL(n.FailedCount,0),
    n.TotalCount,
    n.Details

FROM Numbered n
CROSS JOIN CurrentRun r
CROSS JOIN BaseKey b;

#### DimPatient — only one current version

In [6]:
-- ============================================================
-- DQ: DimPatient
-- ============================================================

WITH CurrentRun AS
(
    SELECT TOP 1 DQRunID
    FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQRun
    WHERE Status = 'RUNNING'
    ORDER BY StartTime DESC
),

BaseKey AS
(
    SELECT ISNULL(MAX(DQResultKey),0) AS BaseKey
    FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQResult
),

Checks AS
(
    SELECT
        'PatientID populated' CheckName,
        'COMPLETENESS' CheckType,
        'CRITICAL' Severity,

        SUM(
            CASE WHEN PatientID IS NULL
                 THEN CAST(1 AS BIGINT)
                 ELSE CAST(0 AS BIGINT)
            END
        ) FailedCount,

        COUNT_BIG(*) TotalCount,

        'PatientID must not be NULL.' Details

    FROM [HP_03_WH_HealthAnalytics].dim.DimPatient


    UNION ALL


    SELECT
        'One current row per PatientID',
        'SCD2',
        'CRITICAL',

        (
            SELECT COUNT_BIG(*)
            FROM
            (
                SELECT PatientID
                FROM [HP_03_WH_HealthAnalytics].dim.DimPatient
                WHERE IsCurrent = 1
                GROUP BY PatientID
                HAVING COUNT_BIG(*) > 1
            ) x
        ),

        (SELECT COUNT_BIG(DISTINCT PatientID)
         FROM [HP_03_WH_HealthAnalytics].dim.DimPatient),

        'A patient may have only one current SCD2 version.'


    UNION ALL


    SELECT
        'ValidTo greater than ValidFrom',
        'SCD2',
        'CRITICAL',

        COUNT_BIG(
            CASE
                WHEN ValidTo <= ValidFrom
                THEN 1
            END
        ),

        COUNT_BIG(*),

        'SCD2 ValidTo must be greater than ValidFrom.'

    FROM [HP_03_WH_HealthAnalytics].dim.DimPatient
),

Numbered AS
(
    SELECT *,
        ROW_NUMBER() OVER (ORDER BY CheckName) rn
    FROM Checks
)

INSERT INTO [HP_09_WH_DataQuality].[dq].SilverToGoldDQResult
SELECT
    b.BaseKey + n.rn,
    r.DQRunID,
    SYSUTCDATETIME(),

    'Gold',
    'DimPatient',

    n.CheckName,
    n.CheckType,
    n.Severity,

    CASE WHEN ISNULL(n.FailedCount,0)=0
         THEN 'PASS' ELSE 'FAIL' END,

    ISNULL(n.FailedCount,0),
    n.TotalCount,
    n.Details

FROM Numbered n
CROSS JOIN CurrentRun r
CROSS JOIN BaseKey b;

##### DimFacility

In [7]:
-- ============================================================
-- DQ: DimFacility
-- ============================================================

WITH CurrentRun AS
(
    SELECT TOP 1 DQRunID
    FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQRun
    WHERE Status='RUNNING'
    ORDER BY StartTime DESC
),

BaseKey AS
(
    SELECT ISNULL(MAX(DQResultKey),0) BaseKey
    FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQResult
),

Checks AS
(
    SELECT
        'FacilityID populated' CheckName,
        'COMPLETENESS' CheckType,
        'CRITICAL' Severity,

        SUM(
            CASE WHEN FacilityID IS NULL
                 THEN CAST(1 AS BIGINT)
                 ELSE CAST(0 AS BIGINT)
            END
        ) FailedCount,

        COUNT_BIG(*) TotalCount,

        'FacilityID must be populated.' Details

    FROM [HP_03_WH_HealthAnalytics].dim.DimFacility


    UNION ALL


    SELECT
        'One current row per FacilityID',
        'SCD2',
        'CRITICAL',

        (
            SELECT COUNT_BIG(*)
            FROM
            (
                SELECT FacilityID
                FROM [HP_03_WH_HealthAnalytics].dim.DimFacility
                WHERE IsCurrent=1
                GROUP BY FacilityID
                HAVING COUNT_BIG(*) > 1
            ) x
        ),

        (SELECT COUNT_BIG(DISTINCT FacilityID)
         FROM [HP_03_WH_HealthAnalytics].dim.DimFacility),

        'Facility must have no more than one current version.'
),

Numbered AS
(
    SELECT *,
        ROW_NUMBER() OVER(ORDER BY CheckName) rn
    FROM Checks
)

INSERT INTO [HP_09_WH_DataQuality].[dq].SilverToGoldDQResult
SELECT
    b.BaseKey+n.rn,
    r.DQRunID,
    SYSUTCDATETIME(),

    'Gold',
    'DimFacility',

    n.CheckName,
    n.CheckType,
    n.Severity,

    CASE WHEN ISNULL(n.FailedCount,0)=0
         THEN 'PASS' ELSE 'FAIL' END,

    ISNULL(n.FailedCount,0),
    n.TotalCount,
    n.Details

FROM Numbered n
CROSS JOIN CurrentRun r
CROSS JOIN BaseKey b;

##### DimProvider DQ check

In [8]:
-- ============================================================
-- DQ: DimProvider
-- ============================================================

WITH CurrentRun AS
(
    SELECT TOP 1 DQRunID
    FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQRun
    WHERE Status='RUNNING'
    ORDER BY StartTime DESC
),

BaseKey AS
(
    SELECT ISNULL(MAX(DQResultKey),0) BaseKey
    FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQResult
),

Checks AS
(
    SELECT
        'ProviderID populated' CheckName,
        'COMPLETENESS' CheckType,
        'CRITICAL' Severity,

        SUM(
            CASE WHEN ProviderID IS NULL
                 THEN CAST(1 AS BIGINT)
                 ELSE CAST(0 AS BIGINT)
            END
        ) FailedCount,

        COUNT_BIG(*) TotalCount,

        'ProviderID must be populated.' Details

    FROM [HP_03_WH_HealthAnalytics].dim.DimProvider


    UNION ALL


    SELECT
        'One current row per ProviderID',
        'SCD2',
        'CRITICAL',

        (
            SELECT COUNT_BIG(*)
            FROM
            (
                SELECT ProviderID
                FROM [HP_03_WH_HealthAnalytics].dim.DimProvider
                WHERE IsCurrent=1
                GROUP BY ProviderID
                HAVING COUNT_BIG(*) > 1
            ) x
        ),

        (SELECT COUNT_BIG(DISTINCT ProviderID)
         FROM [HP_03_WH_HealthAnalytics].dim.DimProvider),

        'Provider must have no more than one current version.'
),

Numbered AS
(
    SELECT *,
        ROW_NUMBER() OVER(ORDER BY CheckName) rn
    FROM Checks
)

INSERT INTO [HP_09_WH_DataQuality].[dq].SilverToGoldDQResult
SELECT
    b.BaseKey+n.rn,
    r.DQRunID,
    SYSUTCDATETIME(),

    'Gold',
    'DimProvider',

    n.CheckName,
    n.CheckType,
    n.Severity,

    CASE WHEN ISNULL(n.FailedCount,0)=0
         THEN 'PASS' ELSE 'FAIL' END,

    ISNULL(n.FailedCount,0),
    n.TotalCount,
    n.Details

FROM Numbered n
CROSS JOIN CurrentRun r
CROSS JOIN BaseKey b;

##### DimPayer DQ check

In [9]:
-- ============================================================
-- DQ: DimPayer
-- ============================================================

WITH CurrentRun AS
(
    SELECT TOP 1 DQRunID
    FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQRun
    WHERE Status='RUNNING'
    ORDER BY StartTime DESC
),

BaseKey AS
(
    SELECT ISNULL(MAX(DQResultKey),0) BaseKey
    FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQResult
),

Checks AS
(
    SELECT
        'PayerID populated' CheckName,
        'COMPLETENESS' CheckType,
        'CRITICAL' Severity,

        SUM(
            CASE WHEN PayerID IS NULL
                 THEN CAST(1 AS BIGINT)
                 ELSE CAST(0 AS BIGINT)
            END
        ) FailedCount,

        COUNT_BIG(*) TotalCount,

        'PayerID must be populated.' Details

    FROM [HP_03_WH_HealthAnalytics].dim.DimPayer


    UNION ALL


    SELECT
        'One current row per PayerID',
        'SCD2',
        'CRITICAL',

        (
            SELECT COUNT_BIG(*)
            FROM
            (
                SELECT PayerID
                FROM [HP_03_WH_HealthAnalytics].dim.DimPayer
                WHERE IsCurrent=1
                GROUP BY PayerID
                HAVING COUNT_BIG(*) > 1
            ) x
        ),

        (SELECT COUNT_BIG(DISTINCT PayerID)
         FROM [HP_03_WH_HealthAnalytics].dim.DimPayer),

        'Payer may have only one current version.'
),

Numbered AS
(
    SELECT *,
        ROW_NUMBER() OVER(ORDER BY CheckName) rn
    FROM Checks
)

INSERT INTO [HP_09_WH_DataQuality].[dq].SilverToGoldDQResult
SELECT
    b.BaseKey+n.rn,
    r.DQRunID,
    SYSUTCDATETIME(),

    'Gold',
    'DimPayer',

    n.CheckName,
    n.CheckType,
    n.Severity,

    CASE WHEN ISNULL(n.FailedCount,0)=0
         THEN 'PASS' ELSE 'FAIL' END,

    ISNULL(n.FailedCount,0),
    n.TotalCount,
    n.Details

FROM Numbered n
CROSS JOIN CurrentRun r
CROSS JOIN BaseKey b;

##### DimDiagnosis DQ check

In [10]:
-- ============================================================
-- DQ: DimDiagnosis
-- ============================================================

WITH CurrentRun AS
(
    SELECT TOP 1 DQRunID
    FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQRun
    WHERE Status='RUNNING'
    ORDER BY StartTime DESC
),

BaseKey AS
(
    SELECT ISNULL(MAX(DQResultKey),0) BaseKey
    FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQResult
),

Checks AS
(
    SELECT
        'DiagnosisCode populated' CheckName,
        'COMPLETENESS' CheckType,
        'CRITICAL' Severity,

        SUM(
            CASE WHEN DiagnosisCode IS NULL
                 THEN CAST(1 AS BIGINT)
                 ELSE CAST(0 AS BIGINT)
            END
        ) FailedCount,

        COUNT_BIG(*) TotalCount,

        'DiagnosisCode must be populated.' Details

    FROM [HP_03_WH_HealthAnalytics].dim.DimDiagnosis


    UNION ALL


    SELECT
        'DiagnosisCode unique',
        'UNIQUENESS',
        'CRITICAL',

        (
            SELECT COUNT_BIG(*)
            FROM
            (
                SELECT DiagnosisCode
                FROM [HP_03_WH_HealthAnalytics].dim.DimDiagnosis
                GROUP BY DiagnosisCode
                HAVING COUNT_BIG(*) > 1
            ) x
        ),

        (SELECT COUNT_BIG(*) FROM [HP_03_WH_HealthAnalytics].dim.DimDiagnosis),

        'DiagnosisCode must be unique.'
),

Numbered AS
(
    SELECT *,
        ROW_NUMBER() OVER(ORDER BY CheckName) rn
    FROM Checks
)

INSERT INTO [HP_09_WH_DataQuality].[dq].SilverToGoldDQResult
SELECT
    b.BaseKey+n.rn,
    r.DQRunID,
    SYSUTCDATETIME(),

    'Gold',
    'DimDiagnosis',

    n.CheckName,
    n.CheckType,
    n.Severity,

    CASE WHEN ISNULL(n.FailedCount,0)=0
         THEN 'PASS' ELSE 'FAIL' END,

    ISNULL(n.FailedCount,0),
    n.TotalCount,
    n.Details

FROM Numbered n
CROSS JOIN CurrentRun r
CROSS JOIN BaseKey b;

##### DimMedication DQ check

In [11]:
-- ============================================================
-- DQ: DimMedication
-- ============================================================

WITH CurrentRun AS
(
    SELECT TOP 1
        DQRunID

    FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQRun

    WHERE Status = 'RUNNING'

    ORDER BY StartTime DESC
),

BaseKey AS
(
    SELECT
        ISNULL(MAX(DQResultKey), 0) AS BaseKey

    FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQResult
),

Checks
(
    CheckName,
    CheckType,
    Severity,
    FailedCount,
    TotalCount,
    Details
)
AS
(

    -- ========================================================
    -- MedicationCode populated
    -- ========================================================

    SELECT

        'MedicationCode populated',

        'COMPLETENESS',

        'CRITICAL',

        SUM(
            CASE

                WHEN MedicationCode IS NULL
                THEN CAST(1 AS BIGINT)

                ELSE CAST(0 AS BIGINT)

            END
        ),

        COUNT_BIG(*),

        'MedicationCode must be populated.'

    FROM [HP_03_WH_HealthAnalytics].dim.DimMedication


    UNION ALL


    -- ========================================================
    -- MedicationCode unique
    -- ========================================================

    SELECT

        'MedicationCode unique',

        'UNIQUENESS',

        'CRITICAL',

        (
            SELECT
                COUNT_BIG(*)

            FROM
            (
                SELECT
                    MedicationCode

                FROM [HP_03_WH_HealthAnalytics].dim.DimMedication

                WHERE MedicationCode IS NOT NULL

                GROUP BY MedicationCode

                HAVING COUNT_BIG(*) > 1

            ) AS x
        ),

        (
            SELECT
                COUNT_BIG(*)

            FROM [HP_03_WH_HealthAnalytics].dim.DimMedication
        ),

        'MedicationCode must be unique.'
),

Numbered AS
(
    SELECT
        CheckName,
        CheckType,
        Severity,
        FailedCount,
        TotalCount,
        Details,

        ROW_NUMBER() OVER
        (
            ORDER BY CheckName
        ) AS rn

    FROM Checks
)


INSERT INTO [HP_09_WH_DataQuality].[dq].SilverToGoldDQResult
(
    DQResultKey,
    DQRunID,
    DQRunTime,

    LayerName,
    EntityName,

    CheckName,
    CheckType,

    Severity,
    Status,

    FailedCount,
    TotalCount,

    Details
)

SELECT

    CAST(b.BaseKey AS BIGINT)
    +
    CAST(n.rn AS BIGINT),

    r.DQRunID,

    SYSUTCDATETIME(),

    'Gold',

    'DimMedication',

    n.CheckName,
    n.CheckType,

    n.Severity,

    CASE

        WHEN ISNULL(n.FailedCount, 0) = 0
        THEN 'PASS'

        ELSE 'FAIL'

    END,

    ISNULL(n.FailedCount, 0),

    n.TotalCount,

    n.Details

FROM Numbered AS n

CROSS JOIN CurrentRun AS r

CROSS JOIN BaseKey AS b;

### Fact tables

##### FactEncounter DQ check

In [12]:
-- ============================================================
-- DQ: FactEncounter
-- ============================================================

WITH CurrentRun AS
(
    SELECT TOP 1 DQRunID
    FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQRun
    WHERE Status='RUNNING'
    ORDER BY StartTime DESC
),

BaseKey AS
(
    SELECT ISNULL(MAX(DQResultKey),0) BaseKey
    FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQResult
),

Checks AS
(
    -- Duplicate EncounterID
    SELECT
        'One row per EncounterID' CheckName,
        'GRAIN' CheckType,
        'CRITICAL' Severity,

        (
            SELECT COUNT_BIG(*)
            FROM
            (
                SELECT EncounterID
                FROM [HP_03_WH_HealthAnalytics].fact.FactEncounter
                GROUP BY EncounterID
                HAVING COUNT_BIG(*) > 1
            ) x
        ) FailedCount,

        (SELECT COUNT_BIG(*)
         FROM [HP_03_WH_HealthAnalytics].fact.FactEncounter) TotalCount,

        'Fact grain is one row per EncounterID.' Details


    UNION ALL


    -- Missing PatientKey
    SELECT
        'PatientKey populated',
        'COMPLETENESS',
        'CRITICAL',

        (
            SELECT COUNT_BIG(*)
            FROM [HP_03_WH_HealthAnalytics].fact.FactEncounter
            WHERE PatientKey IS NULL
        ),

        (SELECT COUNT_BIG(*)
         FROM [HP_03_WH_HealthAnalytics].fact.FactEncounter),

        'Every encounter should resolve to DimPatient.'


    UNION ALL


    -- Missing DateKey
    SELECT
        'DateKey populated',
        'COMPLETENESS',
        'CRITICAL',

        (
            SELECT COUNT_BIG(*)
            FROM [HP_03_WH_HealthAnalytics].fact.FactEncounter
            WHERE DateKey IS NULL
        ),

        (SELECT COUNT_BIG(*)
         FROM [HP_03_WH_HealthAnalytics].fact.FactEncounter),

        'Every encounter should resolve to DimDate.'


    UNION ALL


    -- Orphan PatientKey
    SELECT
        'PatientKey valid',
        'REFERENTIAL_INTEGRITY',
        'CRITICAL',

        (
            SELECT COUNT_BIG(*)

            FROM [HP_03_WH_HealthAnalytics].fact.FactEncounter f

            LEFT JOIN [HP_03_WH_HealthAnalytics].dim.DimPatient p
                ON p.PatientKey = f.PatientKey

            WHERE
                f.PatientKey IS NOT NULL
                AND p.PatientKey IS NULL
        ),

        (SELECT COUNT_BIG(*)
         FROM [HP_03_WH_HealthAnalytics].fact.FactEncounter),

        'PatientKey must exist in DimPatient.'


    UNION ALL


    -- Negative measures
    SELECT
        'Encounter measures non-negative',
        'BUSINESS_RULE',
        'HIGH',

        (
            SELECT COUNT_BIG(*)
            FROM [HP_03_WH_HealthAnalytics].fact.FactEncounter

            WHERE
                   LengthOfStayDays < 0
                OR BaseEncounterCost < 0
                OR TotalClaimCost < 0
                OR PayerCoverage < 0
        ),

        (SELECT COUNT_BIG(*)
         FROM [HP_03_WH_HealthAnalytics].fact.FactEncounter),

        'Length of stay and financial values should not be negative.'
),

Numbered AS
(
    SELECT *,
        ROW_NUMBER() OVER(ORDER BY CheckName) rn
    FROM Checks
)

INSERT INTO [HP_09_WH_DataQuality].[dq].SilverToGoldDQResult
SELECT
    b.BaseKey+n.rn,
    r.DQRunID,
    SYSUTCDATETIME(),

    'Gold',
    'FactEncounter',

    n.CheckName,
    n.CheckType,
    n.Severity,

    CASE WHEN ISNULL(n.FailedCount,0)=0
         THEN 'PASS'
         ELSE 'FAIL'
    END,

    ISNULL(n.FailedCount,0),
    n.TotalCount,
    n.Details

FROM Numbered n
CROSS JOIN CurrentRun r
CROSS JOIN BaseKey b;

##### FactCondition DQ check

In [13]:
-- ============================================================
-- DQ: FactCondition
-- ============================================================

WITH CurrentRun AS
(
    SELECT TOP 1
        DQRunID

    FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQRun

    WHERE Status = 'RUNNING'

    ORDER BY StartTime DESC
),

BaseKey AS
(
    SELECT
        ISNULL(MAX(DQResultKey), 0) AS BaseKey

    FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQResult
),

Checks
(
    CheckName,
    CheckType,
    Severity,
    FailedCount,
    TotalCount,
    Details
)
AS
(
    -- ========================================================
    -- Check 1: One row per ConditionID
    -- ========================================================

    SELECT
        'One row per ConditionID',
        'GRAIN',
        'CRITICAL',

        (
            SELECT COUNT_BIG(*)

            FROM
            (
                SELECT
                    ConditionID

                FROM [HP_03_WH_HealthAnalytics].fact.FactCondition

                WHERE ConditionID IS NOT NULL

                GROUP BY ConditionID

                HAVING COUNT_BIG(*) > 1
            ) AS x
        ),

        (
            SELECT COUNT_BIG(*)
            FROM [HP_03_WH_HealthAnalytics].fact.FactCondition
        ),

        'Fact grain must be one row per ConditionID.'


    UNION ALL


    -- ========================================================
    -- Check 2: PatientKey populated
    -- ========================================================

    SELECT
        'PatientKey populated',
        'COMPLETENESS',
        'CRITICAL',

        (
            SELECT COUNT_BIG(*)

            FROM [HP_03_WH_HealthAnalytics].fact.FactCondition

            WHERE PatientKey IS NULL
        ),

        (
            SELECT COUNT_BIG(*)
            FROM [HP_03_WH_HealthAnalytics].fact.FactCondition
        ),

        'PatientKey must be populated.'


    UNION ALL


    -- ========================================================
    -- Check 3: DiagnosisKey populated
    -- ========================================================

    SELECT
        'DiagnosisKey populated',
        'COMPLETENESS',
        'HIGH',

        (
            SELECT COUNT_BIG(*)

            FROM [HP_03_WH_HealthAnalytics].fact.FactCondition

            WHERE DiagnosisKey IS NULL
        ),

        (
            SELECT COUNT_BIG(*)
            FROM [HP_03_WH_HealthAnalytics].fact.FactCondition
        ),

        'Condition should resolve to DimDiagnosis.'
),

Numbered AS
(
    SELECT
        CheckName,
        CheckType,
        Severity,
        FailedCount,
        TotalCount,
        Details,

        ROW_NUMBER() OVER
        (
            ORDER BY CheckName
        ) AS rn

    FROM Checks
)


INSERT INTO [HP_09_WH_DataQuality].[dq].SilverToGoldDQResult
(
    DQResultKey,
    DQRunID,
    DQRunTime,

    LayerName,
    EntityName,

    CheckName,
    CheckType,

    Severity,
    Status,

    FailedCount,
    TotalCount,

    Details
)

SELECT
    CAST(b.BaseKey AS BIGINT)
    +
    CAST(n.rn AS BIGINT),

    r.DQRunID,

    SYSUTCDATETIME(),

    'Gold',
    'FactCondition',

    n.CheckName,
    n.CheckType,

    n.Severity,

    CASE
        WHEN ISNULL(n.FailedCount, 0) = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END,

    ISNULL(n.FailedCount, 0),

    n.TotalCount,

    n.Details

FROM Numbered AS n

CROSS JOIN CurrentRun AS r

CROSS JOIN BaseKey AS b;

##### 

##### FactReadmission DQ check

In [14]:
-- ============================================================
-- DQ: FactReadmission
-- ============================================================

WITH CurrentRun AS
(
    SELECT TOP 1 DQRunID
    FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQRun
    WHERE Status='RUNNING'
    ORDER BY StartTime DESC
),

BaseKey AS
(
    SELECT ISNULL(MAX(DQResultKey),0) BaseKey
    FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQResult
),

Checks (
    CheckName,
    CheckType,
    Severity,
    FailedCount,
    TotalCount,
    Details

) AS
(
    SELECT
        'PatientKey populated',
        'COMPLETENESS',
        'CRITICAL',

        (SELECT COUNT_BIG(*)
         FROM [HP_03_WH_HealthAnalytics].fact.FactReadmission
         WHERE PatientKey IS NULL),

        (SELECT COUNT_BIG(*)
         FROM [HP_03_WH_HealthAnalytics].fact.FactReadmission),

        'Readmission must resolve to a patient.'


    UNION ALL


    SELECT
        'IndexEncounterKey populated',
        'COMPLETENESS',
        'CRITICAL',

        (SELECT COUNT_BIG(*)
         FROM [HP_03_WH_HealthAnalytics].fact.FactReadmission
         WHERE IndexEncounterKey IS NULL),

        (SELECT COUNT_BIG(*)
         FROM [HP_03_WH_HealthAnalytics].fact.FactReadmission),

        'Index encounter must be populated.'


    UNION ALL


    SELECT
        'ReadmissionEncounterKey populated',
        'COMPLETENESS',
        'CRITICAL',

        (SELECT COUNT_BIG(*)
         FROM [HP_03_WH_HealthAnalytics].fact.FactReadmission
         WHERE ReadmissionEncounterKey IS NULL),

        (SELECT COUNT_BIG(*)
         FROM [HP_03_WH_HealthAnalytics].fact.FactReadmission),

        'Readmission encounter must be populated.'


    UNION ALL


    SELECT
        'DaysToReadmission positive',
        'BUSINESS_RULE',
        'HIGH',

        (SELECT COUNT_BIG(*)
         FROM [HP_03_WH_HealthAnalytics].fact.FactReadmission
         WHERE DaysToReadmission <= 0),

        (SELECT COUNT_BIG(*)
         FROM [HP_03_WH_HealthAnalytics].fact.FactReadmission),

        'DaysToReadmission should be greater than zero.'
),

Numbered AS
(
    SELECT *,
        ROW_NUMBER() OVER(ORDER BY CheckName) rn
    FROM Checks
)

INSERT INTO [HP_09_WH_DataQuality].[dq].SilverToGoldDQResult
SELECT
    b.BaseKey+n.rn,
    r.DQRunID,
    SYSUTCDATETIME(),
    'Gold',
    'FactReadmission',
    n.CheckName,
    n.CheckType,
    n.Severity,
    CASE WHEN ISNULL(n.FailedCount,0)=0
         THEN 'PASS' ELSE 'FAIL' END,
    ISNULL(n.FailedCount,0),
    n.TotalCount,
    n.Details
FROM Numbered n
CROSS JOIN CurrentRun r
CROSS JOIN BaseKey b;

### Summary

In [15]:
-- ============================================================
-- FINAL SILVER -> GOLD DQ SUMMARY
-- ============================================================

WITH CurrentRun AS
(
    SELECT TOP 1
        DQRunID
    FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQRun
    WHERE Status = 'RUNNING'
    ORDER BY StartTime DESC
)

SELECT
    q.EntityName,
    q.CheckName,
    q.CheckType,
    q.Severity,
    q.Status,
    q.FailedCount,
    q.TotalCount,
    q.Details

FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQResult q

INNER JOIN CurrentRun r
    ON r.DQRunID = q.DQRunID

ORDER BY
    CASE q.Severity
        WHEN 'CRITICAL' THEN 1
        WHEN 'HIGH' THEN 2
        WHEN 'MEDIUM' THEN 3
        ELSE 4
    END,

    q.EntityName,
    q.CheckName;

#### Close DQ check

In [16]:
-- ============================================================
-- Close current DQ run
-- ============================================================

UPDATE [HP_09_WH_DataQuality].[dq].SilverToGoldDQRun

SET
    EndTime = SYSUTCDATETIME(),

    Status =
    CASE

        WHEN EXISTS
        (
            SELECT 1

            FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQResult q

            WHERE
                q.DQRunID =
                [HP_09_WH_DataQuality].[dq].SilverToGoldDQRun.DQRunID

                AND q.Severity = 'CRITICAL'

                AND q.Status = 'FAIL'
        )

        THEN 'FAILED'

        ELSE 'PASSED'

    END

WHERE
    Status = 'RUNNING';

In [17]:
SELECT TOP 1
    DQRunID,
    StartTime,
    EndTime,
    Status

FROM [HP_09_WH_DataQuality].[dq].SilverToGoldDQRun

ORDER BY StartTime DESC;